# Managing Cloud Storage from the Command Line


```{admonition} Overview
:class: tip

**Teaching:** 20 min

**Exercises:** 2 min

**Questions:**
* How do I store data in a Bucket?

**Objectives:**
* Manage buckets using the command line.
* Create a bucket with a sensible configuration.
* Learn to manage data (add, list, retrieve, delete) in a bucket using the command line.
```


Now that Drew understands how to create create buckets using the web console and to use the Google command line tools he is now going to explore managing buckets with the command line.

## Configuration

It is important to verify the Account and Project information every time you start interacting with the cloud.  We also use this opportunity to set the configuration environment variables (`PROJECT`) for the Episode.


In [1]:
gcloud config get-value account

998995103712-compute@developer.gserviceaccount.com


In [2]:
gcloud config get-value project

essentials-learner


In [3]:
PROJECT=$(gcloud config list --format='value(core.project)')
echo $PROJECT

essentials-learner


## Create a Bucket

We will first make a new bucket (mb) with mostly default values.  We will explore the newly created bucket with default values then immediately destroy it.  This is a typical pattern when learning a new service by reducing chances to make a mistake (errors can be hard to understand).  In most cases resources should not be created with all default values.  We will re-create the bucket in the next section with more sensible values.    

We first generate a bucket name and store it in an environment variable for future use.  Bucket names are globally unique, so here we use "essentials"  with your Cloud Shell username (the username part of the Google Account) as a prefix to name the bucket something unique and easy to understand where it came from.

In [4]:
BUCKET="essentials-${USER}-$(date +%F)"
echo $BUCKET

essentials-learner-2022-01-14


In [5]:
gsutil mb "gs://$BUCKET"

Creating gs://essentials-learner-2022-01-14/...


If the bucket creation fails with a "ServiceException: 409 A Cloud Storage bucket named ... already exists." it means that the name is not unique or you have already created the bucket.  You will need to either delete the bucket or choose another name.

*Advanced Note: Bucket creation may fail if you are not using the Cloud Shell and the machine you are using has generic account name such as `pi`, `admin`, or other standard username.  If this is the case just set `BUCKET` to some unique value.*

## Show the Bucket

We will now list (Enumerate) the resource to verify that the resource (bucket) has been created by using the `gsutil ls` command to list all the buckets.

In [6]:
gsutil ls

gs://essentials-learner-2022-01-14/


In this case we see that there is only one bucket and it is the one we just created.  Your project may contain other buckets.

Just like the Linux `ls` command, the `gsutil ls` command has a lot of (similar) options.  Let's get detailed information about the bucket.  Note we must specify the bucket by using the `-b` option to specify the bucket.  Type `gsutil ls --help` for more information on the command and options.

In [7]:
gsutil ls -L -b "gs://$BUCKET"

gs://essentials-learner-2022-01-14/ :
	Storage class:			STANDARD
	Location type:			multi-region
	Location constraint:		US
	Versioning enabled:		None
	Logging configuration:		None
	Website configuration:		None
	CORS configuration: 		None
	Lifecycle configuration:	None
	Requester Pays enabled:		None
	Labels:				None
	Default KMS key:		None
	Time created:			Fri, 14 Jan 2022 22:25:26 GMT
	Time updated:			Fri, 14 Jan 2022 22:25:26 GMT
	Metageneration:			1
	Bucket Policy Only enabled:	False
	Public access prevention:	inherited
	RPO:				DEFAULT
	ACL:				
	  [
	    {
	      "entity": "project-owners-998995103712",
	      "projectTeam": {
	        "projectNumber": "998995103712",
	        "team": "owners"
	      },
	      "role": "OWNER"
	    },
	    {
	      "entity": "project-editors-998995103712",
	      "projectTeam": {
	        "projectNumber": "998995103712",
	        "team": "editors"
	      },
	      "role": "OWNER"
	    },
	    {
	      "entity": "project-viewers-998995103712",
	      

You can see that the bucket is "multi-region" and uses the "standard" storage class (this is the type of storage).  The standard storage class is best used for frequently used buckets.  Other storage classes are designed for less frequent use, are less expensive, but come with more restrictions and more complex access costs and should only be used after careful consideration.

Ignore the "ACL" section for now.

You may also want to verify that you can see the newly created bucket in the web console dashboard or "Cloud Storage" page and explore the properties there.

In [8]:
sleep 2 # used to ensure that the next command runs correctly in Jupyter

## Bucket Activity
Next we will check/follow the Activity log to ensure that the bucket was created.  This command assumes that there was no other activity in the account, you may need to increase the `--limit` value to something larger to find the activity.  

Activity logs are used to track important project level activity (such as bucket creation and deletion) and can be used for security and tracking resources and cannot be deleted.  This can be used to debug if something goes wrong or in a security audit or investigation.


In [11]:
gcloud logging read --limit 1

---
insertId: 1dyc72ccp1b
logName: projects/essentials-learner/logs/cloudaudit.googleapis.com%2Factivity
protoPayload:
  '@type': type.googleapis.com/google.cloud.audit.AuditLog
  authenticationInfo:
    principalEmail: 998995103712-compute@developer.gserviceaccount.com
    principalSubject: serviceAccount:998995103712-compute@developer.gserviceaccount.com
    serviceAccountDelegationInfo:
    - firstPartyPrincipal:
        principalEmail: service-998995103712@compute-system.iam.gserviceaccount.com
  authorizationInfo:
  - granted: true
    permission: serviceusage.services.enable
    resource: projectnumbers/998995103712
    resourceAttributes: {}
  - granted: true
    permission: serviceusage.services.enable
    resource: projectnumbers/998995103712
    resourceAttributes: {}
  - granted: true
    permission: serviceusage.operations.get
    resource: projectnumbers/998995103712/operations/-
    resourceAttributes: {}
  - granted: true
    permission: serviceusage.services.get
    res

````{admonition} Note
:class: note

You will probably need to enable the logging api to continue, press `y` to accept as shown below:
```
API [logging.googleapis.com] not enabled on project [998995103712]. Would you 
like to enable and retry (this will take a few minutes)? (y/N)?
```

````

## Use the Bucket

Once the bucket is created, we can now store objects in them and work with them.  It is possible to copy objects (files, strings, etc) in and out of the bucket using the `gsutil cp` command.  The bucket can take objects from the command line or local files

First create a simple file called `one.txt` with the contents "test one"

In [13]:
echo "test one" > one.txt

Display the contents of the file by using the `cat` command.

In [14]:
cat one.txt

test one


Copy the file `one.txt` into the bucket as object "1".

In [15]:
gsutil cp one.txt "gs://$BUCKET/1"

Copying file://one.txt [Content-Type=text/plain]...
/ [1 files][    9.0 B/    9.0 B]                                                
Operation completed over 1 objects/9.0 B.                                        


In [16]:
gsutil ls "gs://$BUCKET"

gs://essentials-learner-2022-01-14/1


Copy the object to the standard out (displays it on the screen).

Remove the orignal file to make sure we are actually seeing the object in the bucket, not the local file.

In [17]:
rm -v one.txt

removed 'one.txt'


In [18]:
gsutil cat "gs://$BUCKET/1"

test one


We can also copy an object to a local file.

In [19]:
gsutil cp "gs://$BUCKET/1" one.txt

Copying gs://essentials-learner-2022-01-14/1...
/ [1 files][    9.0 B/    9.0 B]                                                
Operation completed over 1 objects/9.0 B.                                        


In [20]:
cat one.txt

test one


Remove the local copy.

In [21]:
rm -v one.txt

removed 'one.txt'


We can also copy the output of a command directly into an object in a bucket.  Since output of the `date` command changes we can be sure that we are not seeing old data in an object.

In [22]:
date | gsutil cp - "gs://$BUCKET/2"

Copying from <STDIN>...
/ [1 files][    0.0 B/    0.0 B]                                                
Operation completed over 1 objects.                                              


List all the objects in the buckets and verify the contents of object "2".

In [23]:
gsutil ls "gs://$BUCKET"

gs://essentials-learner-2022-01-14/1
gs://essentials-learner-2022-01-14/2


```{admonition} Exercise

* Display the date in the bucket.
```

In [24]:
gsutil cat "gs://$BUCKET/2"

Fri Jan 14 22:35:35 UTC 2022


Objects can also be removed from buckets.  We will remove object "1" from the Bucket.

In [25]:
gsutil rm "gs://$BUCKET/1"

Removing gs://essentials-learner-2022-01-14/1...
/ [1 objects]                                                                   
Operation completed over 1 objects.                                              


In [26]:
gsutil ls "gs://$BUCKET"

gs://essentials-learner-2022-01-14/2


Try to remove the bucket.

In [27]:
gsutil rb "gs://$BUCKET"
/bin/true # ignore expected error in Jupyter

Removing gs://essentials-learner-2022-01-14/...
NotEmptyException: 409 BucketNotEmpty (essentials-learner-2022-01-14)


Buckets must be empty before they can be deleted (just like subdirectories). (The `/bin/true` ignores the expected error in Jupter so Jupyter does not stop processing the entire notebook) 

In [28]:
gsutil rm "gs://$BUCKET/2"

Removing gs://essentials-learner-2022-01-14/2...
/ [1 objects]                                                                   
Operation completed over 1 objects.                                              


In [29]:
gsutil ls

gs://essentials-learner-2022-01-14/


## Remove the Bucket
Since we are done exploring the bucket we will remove the bucket (rb).  This is a common pattern in cloud computing, to remove resources once we are done with them otherwise they will just sit around incurring costs.

In [30]:
gsutil ls

gs://essentials-learner-2022-01-14/


We first verify that our environment variable is set.  This is a useful pattern to catch simple bugs faster and to aid in debugging.

In [31]:
echo "Bucket: $BUCKET"

Bucket: essentials-learner-2022-01-14


In [32]:
gsutil rb "gs://$BUCKET"

Removing gs://essentials-learner-2022-01-14/...


We verify that the bucket has been removed.  In this example there is no output since there are no more buckets.

In [33]:
gsutil ls

In this case the empty response indicates that there are no Buckets in the project.